In [ ]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = ""
import jax

%load_ext autoreload
%autoreload 2

In [ ]:
import keypoint_moseq as kpms
import logging

# Clear any existing handlers (important in interactive environments)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configure logging once
logging.basicConfig(
    filename='model_run.log',
    level=logging.INFO,
    format='%(asctime)s:%(levelname)s:%(message)s'
)
logging.info("Test log: Notebook execution started.")

project_dir = '/home/fsp585/Desktop/FiberphotometryCode/tests/kpms_session_folders/kpms_3d_test'
config = lambda: kpms.load_config(project_dir)

In [ ]:
from tqdm.notebook import tqdm
from jax_moseq.utils import set_mixed_map_iters
set_mixed_map_iters(32)

In [ ]:
kpms.setup_project(project_dir)

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# # Import our new modular functions.
# from update_config import update_config  # now returns new_mask
# from trimming import create_trimmed_coords_and_confs
# from batching import create_batches

In [ ]:
keypoint_data_path_2d = '/home/fsp585/Desktop/jonathan_ifdn_3d_poses/SplitWithGPU_link/Arena1234_CenterTopCamVid_2024-12-16T11_27_19_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020.h5'
is_2d = False
if is_2d:
    keypoint_data_path = keypoint_data_path_2d

In [ ]:
keypoint_data_path = '/mnt/hdd/old_projects/jonathan_ifdn_3d_poses/anipose_csvs/anipose_csv_no_smoothing/'

In [ ]:
coordinates, confidences, errors, bodyparts = kpms.load_keypoints(keypoint_data_path, 'anipose')

In [ ]:
# Define the scaling function (exactly as in the notebook)
def p3d_preprocess(coords_to_scale, confs_to_scale): # Renamed args to avoid conflict
    print("INFO: Applying p3d_preprocess scaling (per-session)...")
    scaled_coords_dict = {} # Return a new dict
    conf_threshold = 0.5 # Threshold used inside scaling logic

    for key in coords_to_scale:
        print(f"DEBUG:   Scaling key: {key}")
        coords = coords_to_scale[key].copy()
        # *** Important: Which confidences to use for finding good rows? ***
        # The original notebook used trimmed_confidences, which might have NaNs from correction.
        # Let's use that here for replication.
        confs = confs_to_scale[key]

        if coords is None or coords.shape[0] == 0:
             print(f"WARNING:   Skipping scaling for empty coords: {key}")
             scaled_coords_dict[key] = coords
             continue

        try:
             with np.errstate(invalid='ignore'):
                 good_rows = np.all((~np.isnan(confs)) & (confs >= conf_threshold), axis=1)
             num_good_rows = np.sum(good_rows)
             print(f"DEBUG:   Found {num_good_rows} frames where ALL keypoints meet scaling conf >= {conf_threshold}")
        except Exception as e:
             print(f"ERROR:   Error checking good rows for scaling key {key}: {e}")
             good_rows = np.zeros(coords.shape[0], dtype=bool)
             num_good_rows = 0

        if num_good_rows > 0: # Use num_good_rows check
            good_values = coords[good_rows]
            print(f"DEBUG:   Calculating percentiles from shape: {good_values.shape}")
            try:
                lower_percentile = np.nanpercentile(good_values, 0.1, axis=(0, 1))
                upper_percentile = np.nanpercentile(good_values, 99.9, axis=(0, 1))
                if np.any(np.isnan(lower_percentile)) or np.any(np.isnan(upper_percentile)):
                     print(f"WARNING:   NaNs found in percentiles for key {key}. Skipping scaling.")
                     scaled_coords_dict[key] = coords
                else:
                    ranges = upper_percentile - lower_percentile
                    max_range = np.nanmax(ranges)
                    scale = 2047.0 / max_range if max_range > 1e-6 else 1.0
                    print(f"DEBUG:   Scaling params: Lower={np.round(lower_percentile,3)}, Upper={np.round(upper_percentile,3)}, Scale={scale:.3f}")
                    scaled_coords_dict[key] = (coords - lower_percentile) * scale
            except Exception as e_pct:
                 print(f"ERROR:   Error calculating scaling for key {key}: {e_pct}. Keeping original coords.")
                 scaled_coords_dict[key] = coords
        else:
            print(f"WARNING:   No good rows found for scaling key {key}. Setting coords to ZERO.")
            scaled_coords_dict[key] = np.zeros_like(coords)
        print(f"DEBUG:   Shape after scaling for '{key}': {scaled_coords_dict[key].shape}, NaNs: {np.sum(np.isnan(scaled_coords_dict[key]))}")
    return scaled_coords_dict # Return the new dict


print("INFO: Applying scaling function p3d_preprocess...")
# Pass the *trimmed* coordinates and *trimmed* confidences to the scaling function
scaled_coordinates = p3d_preprocess(coordinates, confidences)
# We'll use scaled_coordinates and confidences for format_data
print("INFO: Scaling complete.")
# Log shapes after scaling
if scaled_coordinates:
    key_example = list(scaled_coordinates.keys())[0]
    print(f"DEBUG: Example Shape after scaling (key='{key_example}'): Coords={scaled_coordinates[key_example].shape}")

In [ ]:
import pickle
# === Build trimmed dicts from used-frames pkl (after you've created `scaled_coordinates` & `confidences`) ===
# Assumes: `np`, `pd`, `scaled_coordinates`, `confidences` already exist.

# 1) Load the used-frames mapping
with open("/home/fsp585/Desktop/FiberphotometryCode/tests/used_frames/dfs_by_fname.pkl",'rb') as f:
    used_frames_by_fname = pickle.load(f)  # dict[str, DataFrame-like], has a 'syllable' column

# 2) Helpers to canonicalize keys so pkl <-> scaled keys match
def _extract_timestamp(s: str):
    # Find a 'T' that starts a timestamp like 'YYYY-MM-DDTHH_MM_SS'
    i0 = 0
    while True:
        i = s.find('T', i0)
        if i == -1:
            return None
        # Heuristic: next char should be a digit; we also need 10 chars before and 8 after
        if i >= 10 and (i + 9) <= len(s) and s[i+1:i+2].isdigit():
            return s[i-10:i+9]  # len 19: YYYY-MM-DD + 'T' + HH_MM_SS
        i0 = i + 1

def _extract_view_from_obj_key(s: str):
    # object_file keys end with '_top_left' / '_bottom_left' / '_bottom_right'
    last = s.rfind('_')
    if last == -1:
        return None
    prev = s.rfind('_', 0, last)
    if prev == -1:
        return None
    v1 = s[prev+1:last]
    v2 = s[last+1:]
    return f"{v1}_{v2}"  # e.g., 'top_left'

def _extract_view_from_scaled_key(s: str):
    # scaled keys look like '...<timestamp>_<side>_<view>_3dpoints'
    suffix = '_3dpoints'
    base = s[:-len(suffix)] if s.endswith(suffix) else s
    last = base.rfind('_')
    if last == -1:
        return None
    prev = base.rfind('_', 0, last)
    if prev == -1:
        return None
    # take the last two segments as the view, ignoring the 'left/right' side token
    v1 = base[prev+1:last]
    v2 = base[last+1:]
    return f"{v1}_{v2}"

def _canon_from_obj_key(s: str):
    ts = _extract_timestamp(s)
    view = _extract_view_from_obj_key(s)
    return f"{ts}__{view}" if (ts and view) else None

def _canon_from_scaled_key(s: str):
    ts = _extract_timestamp(s)
    view = _extract_view_from_scaled_key(s)
    return f"{ts}__{view}" if (ts and view) else None

# 3) Build canonical maps
obj_canon_to_mask = {}
for ok, v in used_frames_by_fname.items():
    canon = _canon_from_obj_key(ok)
    if canon is None:
        continue
    # keep frames where syllable is NOT NaN
    try:
        keep = (~v['syllable'].isna()).to_numpy()
    except Exception:
        # fallback if 'syllable' is array-like but not a Series
        keep = np.asarray(v['syllable'])
        keep = np.isfinite(keep)  # treat finite as "used"
    obj_canon_to_mask[canon] = keep

scaled_canon_to_keys = {}
for sk in scaled_coordinates.keys():
    canon = _canon_from_scaled_key(sk)
    if canon is None:
        continue
    scaled_canon_to_keys.setdefault(canon, []).append(sk)

# 4) Trim coords & confs per matched canonical id (and SAVE final masks)
trimmed_coordinates = {}
trimmed_confidences = {}
trimmed_keep_masks  = {}   # <--- NEW: save final boolean mask per kept key

summary_rows = []

for canon, keys in scaled_canon_to_keys.items():
    mask = obj_canon_to_mask.get(canon, None)

    for sk in keys:
        coords = scaled_coordinates[sk]
        T_coords = coords.shape[0] if coords is not None else 0

        conf = confidences.get(sk, None)
        T_conf = (conf.shape[0] if isinstance(conf, np.ndarray)
                  else (len(conf) if conf is not None else np.nan))

        if mask is None:
            # No mask found -> DROP this entry from trimmed dicts
            summary_rows.append({
                "scaled_key": sk,
                "canon": canon,
                "mask_len": np.nan,
                "T_coords": T_coords,
                "T_conf": T_conf,
                "kept": 0,
                "dropped": T_coords,
                "status": "dropped_no_mask",
            })
            continue  # do NOT add to trimmed_* dicts

        # Length reconcile: clip/pad mask to match coords length (time axis)
        m = mask
        if T_coords != len(m):
            if len(m) > T_coords:
                m = m[:T_coords]
            else:
                # pad with False so extras are dropped
                m = np.pad(m, (0, T_coords - len(m)), constant_values=False)

        kept = int(np.sum(m))
        dropped = int(T_coords - kept)

        # Apply along time axis
        trimmed_coordinates[sk] = coords[m, ...] if (coords is not None and T_coords > 0) else coords

        if conf is not None:
            trimmed_confidences[sk] = conf[m, ...] if isinstance(conf, np.ndarray) else np.asarray(conf)[m]

        # Save the FINAL mask used for this key
        trimmed_keep_masks[sk] = m

        summary_rows.append({
            "scaled_key": sk,
            "canon": canon,
            "mask_len": int(len(mask)),
            "T_coords": T_coords,
            "T_conf": T_conf,
            "kept": kept,
            "dropped": dropped,
            "status": "trimmed",
        })

# 5) (Optional) List object-file entries that never matched any scaled key
unmatched_obj = [ok for ok in used_frames_by_fname.keys() if _canon_from_obj_key(ok) not in scaled_canon_to_keys]

# 6) Display a compact summary
summary_df = pd.DataFrame(summary_rows).sort_values(["status", "canon", "scaled_key"]).reset_index(drop=True)
with pd.option_context("display.max_rows", 200, "display.max_columns", 50, "display.width", 220):
    print("=== Trim summary (per scaled key) ===")
    display(summary_df)
    if len(unmatched_obj):
        print("\n=== Object-file keys with no matching scaled key (by timestamp+view) ===")
        display(pd.DataFrame({"object_key": unmatched_obj}))

print(f"\nKept keys: {len(trimmed_coordinates)} | Dropped (no mask): {sum(summary_df['status'] == 'dropped_no_mask')}")
# Results now include:
# - trimmed_coordinates
# - trimmed_confidences
# - trimmed_keep_masks   <--- boolean mask for each kept key



In [ ]:
# REMOVED EXTERNAL BATCHING - Using format_data's internal batching

# format data for modeling
print("INFO: Formatting data using kpms.format_data...")
current_config = config # Load the config from the project directory
print("DEBUG: Config being passed to format_data:")

# Ensure bodyparts are passed if not in config
if 'bodyparts' not in config() or config()['bodyparts'] is None:
    config()['bodyparts'] = bodyparts
    print("DEBUG: Added loaded bodyparts to config for format_data.")

# Use the scaled coordinates if 3D, otherwise use trimmed coordinates
# Use trimmed_confidences (confidences corresponding to trimmed coordinates)
data, metadata = kpms.format_data(trimmed_coordinates, trimmed_confidences, **config())

# --- ADD PRINTS AFTER format_data ---
print(f"INFO: format_data finished. Output data keys: {list(data.keys())}")
if 'Y' in data:
    print(f"INFO:   Formatted data 'Y' shape: {data['Y'].shape}, Type: {type(data['Y'])}")
    try:
        print(f"INFO:   Formatted data 'Y' NaNs: {int(jax.device_get(np.sum(np.isnan(data['Y']))))} / {data['Y'].size}")
    except:
        print(f"INFO:   Formatted data 'Y' NaNs: {np.sum(np.isnan(data['Y']))} / {data['Y'].size} (failed device_get)")
else:
    print("ERROR: 'Y' key missing from format_data output!")
# --- END OF ADDED PRINTS ---


# Fit PCA
print("INFO: Fitting PCA using kpms.fit_pca...")
# Pass data dictionary and re-loaded config dictionary
pca = kpms.fit_pca(**data, **config())
print("INFO: PCA fitting successful!")
print(f"INFO:   PCA components: {pca.n_components_}, Explained variance ratio (top 8): {np.round(pca.explained_variance_ratio_[:8], 3)}")

# kpms.save_pca(pca, project_dir) # Optional

# Original notebook prints/plots
kpms.print_dims_to_explain_variance(pca, 0.8)
# kpms.plot_scree(pca, project_dir=project_dir)
# kpms.plot_pcs(pca, project_dir=project_dir, **config())
kpms.update_config(project_dir, latent_dim=9) # Don't update config here in test

In [ ]:
import logging
from tqdm import tqdm  # Ensure tqdm is imported if not already
import jax

def run_kpms(model_idx, data, pca, metadata, project_dir, kappa, n_ar_iters, n_fit_iters):
    # OPT_KAPPA = 4.207e+05    kappa = OPT_KAPPA
    logging.info(f"Model {model_idx}: Starting AR-only fitting with kappa={kappa}")

    # --- AR-only fitting ---
    model = kpms.init_model(data, pca=pca, **config(), seed=jax.random.PRNGKey(model_idx))
    model = kpms.update_hypparams(model, kappa=kappa)
    model, model_name = kpms.fit_model(
        model, data, metadata, project_dir,
        ar_only=True, num_iters=n_ar_iters)
    
    logging.info(f"Model {model_idx}: Completed AR-only fitting for model '{model_name}'. Loading checkpoint.")
    # Load the checkpoint after AR fitting.
    model, data, metadata, current_iter = kpms.load_checkpoint(
        project_dir, model_name, iteration=n_ar_iters)
    
    # --- Non-AR fitting ---
    logging.info(f"Model {model_idx}: Starting non-AR fitting with adjusted kappa={kappa/10}.")
    model = kpms.update_hypparams(model, kappa=kappa/10)
    model_name = f'kappa_opt_2d_multiple{kappa:.3e}_{model_idx}'
    model = kpms.fit_model(
        model, data, metadata, project_dir, model_name,
        ar_only=False, start_iter=current_iter, num_iters=current_iter + n_fit_iters)[0]
    logging.info(f"Model {model_idx}: Completed non-AR fitting. Final model name: '{model_name}'.")
    kpms.reindex_syllables_in_checkpoint(project_dir, model_name)
    
    return model, model_name

In [ ]:
# Main loop over models

# for i in range(10):
#     kappa = 1e6
#     model, model_name = run_kpms(i, data, pca, metadata, project_dir, kappa, 50, 250)
    
    # # Process batches and apply model
    # for i, (coords_batch, confs_batch) in enumerate(tqdm(zip(batches_coords, batches_confs), total=len(batches_coords))):
    #     logging.info(f"Model '{model_name}': Processing batch {i}")
    #     data_batch, metadata_batch = kpms.format_data(coords_batch, confs_batch, **config())
    #     results = kpms.apply_model(model, data_batch, metadata_batch, project_dir, model_name, **config())
        
    #     # Append the batch index to the save directory to make it unique.
    #     save_directory = f"{project_dir}/{model_name}/results_batch_{i}"
    #     kpms.save_results_as_csv(results, project_dir, model_name, save_dir=save_directory)
    #     logging.info(f"Model '{model_name}': Saved results for batch {i} to '{save_directory}'")
    
    # logging.info(f"Completed all batches for model '{model_name}'")


In [ ]:
model_name_prefix = 'kappa_opt_2d_multiple1.000e+06'
model_names = [f'{model_name_prefix}_{i}' for i in range(8)]

In [ ]:
import os
import logging
from tqdm import tqdm
import numpy as np

# Option A: Use the vectorized function, wrapping the call in tqdm for a progress update.
eml_scores, eml_std_errs = kpms.expected_marginal_likelihoods(project_dir, model_names)

# Identify the best model (highest EML score)
best_model = model_names[np.argmax(eml_scores)]
logging.info(f"Best model: {best_model} (params)")

# Plot the EML scores for all models
kpms.plot_eml_scores(eml_scores, eml_std_errs, model_names)


Bedste model er *multiple1.000e+06_3

In [ ]:
# for curr_model in curr_models:
for model_name in tqdm(model_names):
    model, data, metadata, current_iter = kpms.load_checkpoint(project_dir, model_name)
    results = kpms.extract_results(model, metadata, project_dir, model_name)
    kpms.save_results_as_csv(results, project_dir, model_name)
    kpms.reindex_syllables_in_checkpoint(project_dir, model_name)
    kpms.generate_trajectory_plots(trimmed_coordinates, results, project_dir, model_name, fps=80, 
                                   projection_planes=["xy", "xz", "yz"], **config())
    # kpms.plot_similarity_dendrogram(init_coords, results, project_dir, curr_model, **config())


In [ ]:
best_model = model_names[3]

In [ ]:
model, data, metadata, current_iter = kpms.load_checkpoint(project_dir, best_model)
results = kpms.extract_results(model, metadata, project_dir, best_model)

In [ ]:
for k, v in results.items():
    print(k, len(v['syllable']), v.keys())

In [ ]:
video_dir = "/mnt/hdd/Jonathan_IFxDN/AllTopcams/SplitWithGPU"

def _search_key(s):
    parts = s.split('_')
    return 'Arena1234_CenterTopCamVid_' +\
          '_'.join(p for i, p in enumerate(parts) if i != 3 and i != len(parts) - 1)

key_map = {k: _search_key(k) for k in results.keys()}  # orig -> transformed
matches = kpms.find_matching_videos(key_map.values(), video_dir, as_dict=True)
video_paths = {k: matches.get(key_map[k]) for k in results.keys()}

In [ ]:
video_paths

In [ ]:
video_frame_indexes = {k: np.flatnonzero(trimmed_keep_masks[k]) for k in trimmed_coordinates.keys()}

In [ ]:
suffix = "DLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020.csv"
keypoint_paths2d = {k: v[:-4] + suffix for k, v in video_paths.items()}

In [ ]:
coordinates_2D,_,_ = kpms.load_keypoints(list(keypoint_paths2d.values()), 'deeplabcut')

In [ ]:
def _coords2d_key(res_key):
    return keypoint_paths2d[res_key].split('/')[-1].rsplit('.', 1)[0]

# Build a trimmed 2D dict keyed by result keys (to match coords/conf dicts)
coordinates_2D_trimmed = {}
for res_key in trimmed_coordinates.keys():
    k2d = _coords2d_key(res_key)
    idx = video_frame_indexes[res_key]              # original-frame indexes kept
    coordinates_2D_trimmed[res_key] = coordinates_2D[k2d][idx, ...]


In [ ]:
centroids, headings = kpms.get_centroids_headings(coordinates_2D_trimmed, **config())

In [ ]:
# make the grid movies
kpms.generate_grid_movies(
   results,
   project_dir,
   best_model,
   video_paths=video_paths,
   coordinates=coordinates_2D_trimmed,
   centroids=centroids,
   headings=headings,
   fps=80,
   video_frame_indexes=video_frame_indexes,
   overlay_keypoints=False,
   **config());

In [ ]:
results['2024-12-13T14_50_39_left_bottom_left_3dpoints']['syllable']

In [ ]:
# Recover pre-trim cardinality into DataFrames per fname using trimmed_keep_masks
# Expects:
#   results: dict[fname -> dict[attr -> np.ndarray post-trim]]
#   trimmed_keep_masks: dict[fname -> boolean mask over original frames]
# Produces:
#   full_dfs: dict[fname -> pd.DataFrame] (NaNs inserted where mask == False)

def results_to_full_dfs(results, keep_masks):
    full = {}
    for fname, inner in results.items():
        m = keep_masks.get(fname)
        if m is None:
            continue  # dropped earlier; no mask available
        T = len(m)
        N = int(m.sum())
        cols = {}
        for attr, arr in inner.items():
            a = np.asarray(arr)
            if a.shape[0] != N:
                raise ValueError(f"{fname}:{attr} length {a.shape[0]} != mask sum {N}")
            if a.ndim == 1:
                if a.dtype.kind in ("U", "S", "O"):
                    col = np.empty(T, dtype=object); col[:] = np.nan
                    col[m] = a
                else:
                    col = np.full(T, np.nan)
                    col[m] = a.astype(float, copy=False)
                cols[attr] = col
            else:
                flat = a.reshape(N, -1)
                for j in range(flat.shape[1]):
                    if a.dtype.kind in ("U", "S", "O"):
                        col = np.empty(T, dtype=object); col[:] = np.nan
                        col[m] = flat[:, j]
                    else:
                        col = np.full(T, np.nan)
                        col[m] = flat[:, j].astype(float, copy=False)
                    cols[f"{attr}_{j}"] = col
        full[fname] = pd.DataFrame(cols)
    return full

full_dfs = results_to_full_dfs(results, trimmed_keep_masks)


In [ ]:
import pickle
import os

outpath = "/home/fsp585/Desktop/FiberphotometryCode/tests/used_frames/dfs_by_name_3d_runs.pkl"
os.makedirs(os.path.dirname(outpath), exist_ok=True)

with open(outpath, "wb") as f:
    pickle.dump(full_dfs, f)